In [1]:
import numpy as np
import pandas as pd
import snowflake.connector
from datetime import datetime, timedelta
import requests
import io
from scipy.optimize import linprog
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value
import os

In [2]:
# Inicializar una lista para almacenar los DataFrames
dataframes = []

# Intentar cargar cada archivo y agregarlo a la lista si está presente
try:
    if os.path.exists(r"C:\Users\juane\Downloads\PLAN_PANADERIA.csv"):
        df_plan_panaderia = pd.read_csv(r"C:\Users\juane\Downloads\PLAN_PANADERIA.csv")
        dataframes.append(df_plan_panaderia)
        print("PLAN_PANADERIA.csv cargado con éxito.")
    else:
        print("PLAN_PANADERIA.csv no encontrado.")

    if os.path.exists(r"C:\Users\juane\Downloads\PLAN_CONGELADOS.csv"):
        df_plan_congelados = pd.read_csv(r"C:\Users\juane\Downloads\PLAN_CONGELADOS.csv")
        dataframes.append(df_plan_congelados)
        print("PLAN_CONGELADOS.csv cargado con éxito.")
    else:
        print("PLAN_CONGELADOS.csv no encontrado.")

    if os.path.exists(r"C:\Users\juane\Downloads\PLAN_COMPRA_GENERAL.csv"):
        df_plan_compra_general = pd.read_csv(r"C:\Users\juane\Downloads\PLAN_COMPRA_GENERAL.csv")
        dataframes.append(df_plan_compra_general)
        print("PLAN_COMPRA_GENERAL.csv cargado con éxito.")
    else:
        print("PLAN_COMPRA_GENERAL.csv no encontrado.")
    
    # Verificar si al menos uno de los archivos fue cargado
    if not dataframes:
        raise FileNotFoundError("Ningún archivo CSV de planes de compra está presente. Proceso detenido.")
    
    # Cocatenar en un solo DataFrame si hay al menos un DataFrame cargado
    df_plan_externos = pd.concat(dataframes, ignore_index=True)
    df_plan_externos = df_plan_externos.reset_index(drop=True)
    print("Archivos concatenados exitosamente.")
    
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

PLAN_PANADERIA.csv no encontrado.
PLAN_CONGELADOS.csv no encontrado.
PLAN_COMPRA_GENERAL.csv cargado con éxito.
Archivos concatenados exitosamente.


In [6]:
#Crear columna warehouse product id
df_plan_externos['WAREHOUSE_PRODUCT_ID'] = df_plan_externos['Warehouseid'].astype(str) + '-' + df_plan_externos['Sku Id'].astype(str)

In [9]:
#Crear un dataframe que contenga un mapeo de warehouse id, city id y city name
warehouse_city_mapping = pd.DataFrame({
    'Warehouseid': [49, 63, 66, 84, 107, 21, 25, 27, 29, 31, 48, 52, 54, 55, 68, 80, 87, 91, 92, 3, 4, 5, 6, 7, 108, 109, 2, 75, 79, 90, 93, 8, 9, 11, 12, 14, 16, 17, 18, 19, 20, 23, 24, 26, 28, 30, 33, 47, 50, 51, 53, 56, 58, 60, 61, 67, 69, 70, 72, 73, 78, 110, 103, 104, 106, 105],
    'CITY_ID': [3, 3, 3, 5, 5, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 35, 35, 35, 35, 35, 35, 35, 36, 36, 36, 36, 36, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 40, 42, 42, 43],
    'CITY_NAME': ['Barranquilla', 'Barranquilla', 'Barranquilla', 'Bucaramanga', 'Bucaramanga', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Cartagena', 'Barranquilla', 'Cartagena', 'Cartagena', 'Cartagena', 'Santa Marta', 'Villavicencio', 'Cali', 'Cali', 'Cali', 'Cali', 'Cali', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Armenia', 'Pereira', 'Ibagué', 'Montería']
})

In [10]:
#Realizar un left join con el dataframe que contiene un mapeo de warehouse id, city id y city name
df_plan_externos = df_plan_externos.merge(warehouse_city_mapping, on='Warehouseid', how='left')

In [12]:
multiples_restrictions = {
    77: 5, #Pan pa ya
    316: 5, #STLTH
    204:5, #La Boutique
    45:5, #New Pork
    210:10, #Pod Salt
    95:1, #Luisa Postres
    168:10, #Glucloud
     4:5, #Inversiones Liev
    200:5, #Arte Dolce
    135: 6, #Sr Wok
    305:12, #Rosadela
    318:10, #Evapify
    185:5, #Relx
    144:12, #Productos Donald's
    313:{ #Notco
        14467:32,
        14469:32,
        14472:18,
        14473:56
        },
    119: { #Pixie
        3555: 20,
        3320: 20,
        3468: 20,
        3587: 20
    }
}

In [13]:
#Redondea las cantidades al multiplo mas cercano (hacia arriba y hacia abajo según sea necesario)
def round_to_multiple(quantity, multiple):
    return round(quantity / multiple) * multiple

In [14]:
#Crear Columna adjusted quantity y asignarle el resultado de redondear los valores de compra al multiplo especificado
def apply_multiples_restrictions(df, multiples_restrictions):
    df['ADJUSTED_QUANTITY'] = df['Purchase Quantity']
    
    for supplier, restriction in multiples_restrictions.items():
        supplier_df = df[df['Primary Supplier Id'] == supplier]
        
        if isinstance(restriction, dict):
            #Proveedores con diferentes multiplos por producto
            for product, multiple in restriction.items():
                product_df = supplier_df[supplier_df['Sku Id'] == product]
                for idx, row in product_df.iterrows():
                    ADJUSTED_QUANTITY = round_to_multiple(row['Purchase Quantity'], multiple)
                    df.loc[idx, 'ADJUSTED_QUANTITY'] = ADJUSTED_QUANTITY
        else:
            #El mismo multiplo para todos los productos
            for idx, row in supplier_df.iterrows():
                ADJUSTED_QUANTITY = round_to_multiple(row['Purchase Quantity'], restriction)
                df.loc[idx, 'ADJUSTED_QUANTITY'] = ADJUSTED_QUANTITY
                
    return df


In [15]:
supplier_restrictions = {
    316: [ #STLTH
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 30, 'max_quantity': 5000}
    ],
    57: [ #Vive Agro
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 25, 'max_quantity': 5000}
    ],
    178: [ #Kampos carnicos
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 10, 'max_quantity': 5000},
    ],
    313: [ #Notco
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 20, 'max_quantity': 5000}
    ],
    166: [ #Selvati
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 20, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 20, 'max_quantity': 5000}
    ],
   210: [ #Pod Salt
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 56, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 50, 'max_quantity': 5000}, 
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 50, 'max_quantity': 5000}
   ],
    168:[ #Glucloud
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 50, 'max_quantity': 5000}
    ],
    4:[ #Inversiones Liev
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 10, 'max_quantity': 5000}
    ],
    200:[ #Arte Dolce
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 40, 'max_quantity': 5000}
    ],
    135: [ #Sr Wok
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 18, 'max_quantity': 5000}
    ],
    185:[ #RELX
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 25, 'max_quantity': 5000}
    ],
    318: [ #Evapify
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 30, 'max_quantity': 5000}
    ],
    77:[ #Pan pa ya
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_value': 70000, 'max_value': 10000000}
    ],
    119 :[ #Pixie
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 35, 'min_value': 800000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 3, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 5, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 7, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 36, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 37, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 40, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 42, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 43, 'min_value': 350000, 'max_value': 10000000}
    ],
    
}
    
        
    

In [16]:
#Levels: product,warehouse,city,total
#types: quantity, monetary

def check_supplier_restrictions(df, supplier_restrictions, multiples_restrictions):
    df['VIOLATION'] = False
    df['VIOLATION_DESCRIPTION'] = ''
    
    # Apply multiples restrictions and adjust quantities
    df = apply_multiples_restrictions(df, multiples_restrictions)
    
    for supplier, restrictions in supplier_restrictions.items():
        supplier_df = df[df['Primary Supplier Id'] == supplier]
        print(f"Checking restrictions for supplier: {supplier}")  # Debug
        print(f"Supplier DataFrame:\n{supplier_df}")  # Debug
        
        if supplier_df.empty:
            print(f"  No data for supplier: {supplier}")  # Debug
            continue
        
        for restriction in restrictions:
            restriction_type = restriction['type']
            level = restriction['level']
            
            if level == 'warehouse':
                group_col = 'Warehouseid'
            elif level == 'city':
                group_col = 'City ID'
            elif level == 'product':
                group_col = 'Sku Id'
            else: 
                group_col = None
                
            if group_col:
                if group_col not in supplier_df.columns:
                    print(f"  Column {group_col} not in supplier DataFrame")  # Debug
                    continue
 
                grouped = supplier_df.groupby(group_col)
                print(f"Grouping by {group_col}: {grouped.groups.keys()}")  # Debug
                
                if not grouped.groups:
                    print(f"  No groups found for {group_col} in supplier DataFrame")  # Debug
                    continue
                
                for key, group in grouped:
                    print(f"    Grouping by {group_col}, Key: {key}")  # Debug
                    if restriction_type == 'quantity':
                        total_quantity = group['ADJUSTED_QUANTITY'].sum()
                        print(f"      Total quantity: {total_quantity}")  # Debug
                        if total_quantity < restriction['min_quantity'] or total_quantity > restriction['max_quantity']:
                            df.loc[group.index, 'VIOLATION'] = True
                            df.loc[group.index, 'VIOLATION_DESCRIPTION'] = f'{restriction_type} violation at {level} level for {group_col}={key}'
                    elif restriction_type == 'monetary':
                        total_value = (group['ADJUSTED_QUANTITY'] * group['Supplier Price Avg']).sum()
                        print(f"      Total value: {total_value}")  # Debug
                        if total_value < restriction['min_value'] or total_value > restriction['max_value']:
                            df.loc[group.index, 'VIOLATION'] = True
                            df.loc[group.index, 'VIOLATION_DESCRIPTION'] = f'{restriction_type} violation at {level} level for {group_col}={key}'
            else: 
                if restriction_type == 'quantity':
                    total_quantity = supplier_df['ADJUSTED_QUANTITY'].sum()
                    print(f"    Total quantity: {total_quantity}")  # Debug
                    if total_quantity < restriction['min_quantity'] or total_quantity > restriction['max_quantity']:
                        df.loc[supplier_df.index, 'violation'] = True
                        df.loc[supplier_df.index, 'violation_description'] = f'{restriction_type} violation at total level'
                elif restriction_type == 'monetary':
                    total_value = (supplier_df['ADJUSTED_QUANTITY'] * supplier_df['PRIMARY_SUPPLIER_PRICE']).sum()
                    print(f"    Total value: {total_value}")  # Debug
                    if total_value < restriction['min_value'] or total_value > restriction['max_value']:
                        df.loc[supplier_df.index, 'violation'] = True
                        df.loc[supplier_df.index, 'violation_description'] = f'{restriction_type} violation at total level'
    
    return df

In [17]:
def optimize_increase(group, required_increment):
    # Extraer datos del grupo
    adjusted_quantity = group['ADJUSTED_QUANTITY'].values
    shelf_life = group['Shelf Life'].values
    current_doh = group['Current DOH'].values

    # Obtener el SKU del primer elemento
    sku_id = group['Sku Id'].iloc[0]

    # Intentar obtener el múltiplo del SKU
    multiples = multiples_restrictions.get(sku_id, 1)
    if multiples == 1:
        print(f"SKU {sku_id} not found in multiples_restrictions. Defaulting to multiple of 1.")

    # Definir el número de productos en el grupo
    num_products = len(adjusted_quantity)

    # Crear el problema de minimización
    prob = LpProblem("OptimizeIncrease", LpMinimize)

    # Crear las variables de ajuste con el objetivo de minimizar el incremento
    adjustments = [LpVariable(f"adjustment_{i}", lowBound=0, cat="Continuous") for i in range(num_products)]

    # Función objetivo: minimizar el ajuste total
    prob += lpSum(adjustments)

    # Restricción de igualdad: la suma de ajustes debe ser igual al incremento requerido
    prob += lpSum(adjustments) == required_increment

    # Restricciones de desigualdad para el shelf life
    for i in range(num_products):
        daily_sales_avg = group['Average Daily Sales for Past 7 Days'].iloc[i]
        if daily_sales_avg > 0:
            max_adjustment = (shelf_life[i] - current_doh[i]) * daily_sales_avg
            prob += adjustments[i] <= max_adjustment

    # Resolver el problema de optimización
    prob.solve()

    # Verificar el resultado
    if LpStatus[prob.status] == 'Optimal':
        # Obtener los valores ajustados y redondearlos al múltiplo
        adjustments_values = [round(value(adj) / multiples) * multiples for adj in adjustments]
        return adjustments_values
    else:
        return None  # Indicar fallo en la optimización


In [18]:
def optimize_decrease(group, excess_amount):
    # Extraer datos del grupo
    adjusted_quantity = group['ADJUSTED_QUANTITY'].values
    current_doh = group['Current DOH'].values

    # Obtener el SKU del primer elemento
    sku_id = group['Sku Id'].iloc[0]

    # Intentar obtener el múltiplo del SKU
    multiples = multiples_restrictions.get(sku_id, 1)
    if multiples == 1:
        print(f"SKU {sku_id} not found in multiples_restrictions. Defaulting to multiple of 1.")

    # Definir el número de productos en el grupo
    num_products = len(adjusted_quantity)

    # Crear el problema de minimización de reducción total
    prob = LpProblem("OptimizeDecrease", LpMinimize)

    # Crear las variables de ajuste
    adjustments = [LpVariable(f"adjustment_{i}", lowBound=0, cat="Continuous") for i in range(num_products)]

    # Función objetivo: minimizar la suma de ajustes
    prob += lpSum(adjustments)

    # Restricción de igualdad: la suma de ajustes debe alcanzar el exceso de cantidad
    prob += lpSum(adjustments) == excess_amount

    # Restricciones de desigualdad: no caer por debajo del mínimo permitido
    for i in range(num_products):
        min_allowed_quantity = group['ADJUSTED_QUANTITY'].iloc[i] - excess_amount
        prob += adjustments[i] <= adjusted_quantity[i] - min_allowed_quantity

    # Resolver el problema de optimización
    prob.solve()

    # Verificar si la solución es óptima
    if LpStatus[prob.status] == 'Optimal':
        # Obtener los valores ajustados y redondearlos al múltiplo
        adjustments_values = [round(value(adj) / multiples) * multiples for adj in adjustments]
        return adjustments_values
    else:
        return None  # Indicar fallo en la optimización

In [19]:
def adjust_quantities(df, supplier_restrictions, multiples_restrictions):
    # Inicializar la columna de estado de ajuste
    df['ADJUSTMENT_STATUS'] = ''
    
    for supplier, restrictions in supplier_restrictions.items():
        print(f"Checking restrictions for supplier: {supplier}")  # debug
        supplier_df = df[df['Primary Supplier Id'] == supplier]
        print(f"Supplier DataFrame shape before filtering: {supplier_df.shape}")  # debug
        
        for restriction in restrictions:
            restriction_type = restriction['type']
            level = restriction['level']

            # Obtener min_value y max_value según el tipo de restricción
            if restriction_type == 'quantity':
                min_value = restriction.get('min_quantity', 0)
                max_value = restriction.get('max_quantity', float('inf'))
            elif restriction_type == 'monetary':
                min_value = restriction.get('min_value', 0)
                max_value = restriction.get('max_value', float('inf'))
            else:
                print(f"Unknown restriction type: {restriction_type}")  # debug
                continue
            
            print(f"Restriction Level: {level}, Min Value: {min_value}, Max Value: {max_value}")  # debug

            # Agrupación según el nivel
            if level == 'warehouse':
                group_col = 'Warehouseid'
            elif level == 'city':
                group_col = 'City ID'
            elif level == 'product':
                group_col = 'Sku Id'
            else:
                group_col = None
            
            if group_col:
                grouped = supplier_df.groupby(group_col)

                for key, group in grouped:
                    total_quantity = group['ADJUSTED_QUANTITY'].sum()
                    total_value = (group['ADJUSTED_QUANTITY'] * group['Supplier Price Avg']).sum()
                    print(f"Group: {key}, Total Quantity: {total_quantity}, Total Value: {total_value}")  # debug
                    
                    # Verificar si necesita ajustes
                    print(f"Comparing: Total Quantity {total_quantity} with Min Value {min_value}")  # debug
                    if total_quantity < min_value:
                        print(f"Adjusting up for group {key}, required increment: {min_value - total_quantity}")  # debug
                        # Ajustar hacia arriba
                        required_increment = min_value - total_quantity
                        adjustments = optimize_increase(group, required_increment)
                        print(f"Adjustments made: {adjustments}")  # debug
                        if adjustments is not None:
                            df.loc[group.index, 'ADJUSTED_QUANTITY'] += adjustments
                        else:
                            df.loc[group.index, 'ADJUSTMENT_STATUS'] = 'Failed to meet min quantity'
                    
                    elif total_value > max_value:
                        print(f"Adjusting down for group {key}, excess amount: {total_value - max_value}")  # debug
                        # Ajustar hacia abajo
                        excess_amount = total_value - max_value
                        adjustments = optimize_decrease(group, excess_amount)
                        print(f"Adjustments made: {adjustments}")  # debug
                        if adjustments is not None:
                            df.loc[group.index, 'ADJUSTED_QUANTITY'] -= adjustments
                        else:
                            df.loc[group.index, 'ADJUSTMENT_STATUS'] = 'Failed to meet max value'

    return df


In [20]:
df_purchase_plans_with_violations = check_supplier_restrictions(df_plan_externos, supplier_restrictions, multiples_restrictions)

Checking restrictions for supplier: 316
Supplier DataFrame:
Empty DataFrame
Columns: [Warehouseid, Sku Id, Sku Name, Purchase Plan Duration Start Date, Purchase Plan Duration End Date, Duration, Duration Forecast(Final), Current Stock, Predicted Inventory At The Start Of Duration, Current DOH, Primary Supplier Id, Primary Supplier Name, Supplier Lead Time (Days), Supplier Price Avg, Supplier Mrp Avg, Supplier Fill Rate Avg, Safety Stock, Safety Stock Days, Lead Time Safety Stock, Duration Safety Stock, Shelf Life, Lead Time Forecast, Average Daily Sales for Past 60 Days, Average Daily Sales for Past 14 Days, Average Daily Sales for Past 7 Days, SKU Type, Batch Size, Incoming Stock, Net Inter-Store Transfers, Expiring Quantity, Minimum Order Quantity, Minimum Presentation Quantity, Purchase Quantity, Purchase Quantity (Min Val Adj.), Purchase Quantity (Min Val Adj.) Editable, Purchase Quantity Editable (Batches), Purchase Quantity Diff., Purchase Quantity % Diff., Purchase Quantity (In 

In [21]:
df_adjusted_plan = adjust_quantities(df_purchase_plans_with_violations, supplier_restrictions, multiples_restrictions)

Checking restrictions for supplier: 316
Supplier DataFrame shape before filtering: (0, 91)
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value: 5000
Restriction Level: warehouse, Min Value: 30, Max Value:

In [22]:
df_adjusted_plan.to_csv(r"C:\Users\juane\Downloads\PURCHASE_PLAN_COLOMBIA.csv", index=False)